# 数据集准备
## 云端数据集导入 → 探索 → 合并

运行前请确认以下数据集已导入到 Mo 平台：
- `/home/jovyan/work/datasets/69f46e75dbb43ba9e05483c1-69e0f1d5638ba61f00d54c83`
- `/home/jovyan/work/datasets/6a11324dca6944e11f9fc58a-6a084256638ba61f00d565fb`

In [ ]:
# Cell 1: 安装依赖
!pip install -r requirements.txt

In [ ]:
# Cell 2: 探索数据集目录结构
import os

datasets_root = "/home/jovyan/work/datasets"
dataset_dirs = [
    "69f46e75dbb43ba9e05483c1-69e0f1d5638ba61f00d54c83",
    "6a11324dca6944e11f9fc58a-6a084256638ba61f00d565fb",
]

for ds_dir in dataset_dirs:
    full_path = os.path.join(datasets_root, ds_dir)
    print(f"{'='*60}")
    print(f"数据集: {ds_dir}")
    print(f"路径: {full_path}")
    print(f"存在: {os.path.exists(full_path)}")

    if not os.path.exists(full_path):
        print("  ⚠ 路径不存在，请确认数据集已导入")
        continue

    # 列出顶层内容
    top_items = sorted(os.listdir(full_path))
    print(f"顶层内容: {top_items}")

    # 递归统计各类别文件数量
    for item in top_items:
        item_path = os.path.join(full_path, item)
        if os.path.isdir(item_path) and not item.startswith('_'):
            # 检查是否为类目录（含图片文件）
            img_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'}
            img_files = [f for f in os.listdir(item_path)
                        if os.path.splitext(f)[1].lower() in img_extensions]
            if img_files:
                print(f"  [{item}] {len(img_files)} 张图片")
            else:
                # 可能是嵌套型（含 train/test 子目录）
                sub_items = sorted(os.listdir(item_path))
                print(f"  [{item}/] 子目录: {sub_items}")
                for sub_item in sub_items:
                    sub_path = os.path.join(item_path, sub_item)
                    if os.path.isdir(sub_path):
                        sub_sub = sorted(os.listdir(sub_path))
                        print(f"    [{sub_item}/] {sub_sub}")
                        for cls_dir_name in sub_sub:
                            cls_path = os.path.join(sub_path, cls_dir_name)
                            if os.path.isdir(cls_path):
                                cls_files = [f for f in os.listdir(cls_path)
                                           if os.path.splitext(f)[1].lower() in img_extensions]
                                if cls_files:
                                    print(f"      [{cls_dir_name}] {len(cls_files)} 张图片")
        elif item.lower().endswith('.zip'):
            size_mb = os.path.getsize(item_path) / (1024 * 1024)
            print(f"  [{item}] zip 文件 ({size_mb:.1f} MB)")

print(f"\n{'='*60}")
print("扫描完成")
print(f"\n提示: 以上信息可用于判断数据集结构是否正确。")
print(f"  - 平铺型: 类名目录直接在顶层 → 可被 auto 模式识别")
print(f"  - 嵌套型: train/test/{class}/ → prepare_data() 自动识别合并")
print(f"  - zip 文件: 自动解压后处理")

In [ ]:
# Cell 3: 检查 auto-discovery 是否能发现这些数据集
import sys
sys.path.insert(0, os.getcwd())

from config import CONFIG

print("当前 data_roots 配置:", CONFIG["data_roots"])
print("目标类别:", CONFIG["class_names"])
print("类名别名表:", CONFIG["class_aliases"])

# 模拟 auto 扫描（不实际复制）
from data.dataset import _scan_datasets_dir, _get_data_roots

entries = _scan_datasets_dir(
    datasets_root=datasets_root,
    target_classes=CONFIG["class_names"],
    aliases=CONFIG["class_aliases"],
)

print(f"\n发现 {len(entries)} 个数据源:")
for i, entry in enumerate(entries):
    print(f"  [{i+1}] {entry['path']}")
    if entry['class_map']:
        print(f"      class_map: {entry['class_map']}")

In [ ]:
# Cell 4: 数据准备 — 合并所有数据源到可写目录
from data.dataset import prepare_data

data_root = prepare_data()
print(f"\n数据已准备完成，路径: {data_root}")

In [ ]:
# Cell 5: 创建 DataLoader 并验证数据划分
from data.dataset import create_dataloaders

train_loader, val_loader, class_counts = create_dataloaders()
print(f"\n训练批次: {len(train_loader)}, 验证批次: {len(val_loader)}")
print(f"批次大小: {CONFIG['batch_size']}")
print(f"\n数据准备验证通过 ✓ — 可以开始训练")

## 数据准备完成后

打开 `coding_here.ipynb` 继续训练管线：

| Cell | 内容 |
|------|------|
| Cell 4 | 训练教师模型 (EfficientNet-B5) |
| Cell 5 | 知识蒸馏 (B5 → B0) |
| Cell 6 | 结构化剪枝 + 微调 |
| Cell 7 | ONNX 导出 + INT8 量化 + 测速 |